In [2]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

DATA_PATH = Path("hour.csv")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["dteday"])
df = df.sort_values(["dteday", "hr"]).reset_index(drop=True)
print(df.shape, df.dteday.min().date(), "->", df.dteday.max().date())

(17379, 17) 2011-01-01 -> 2012-12-31


In [3]:
# 1. Leakage: casual + registered is exactly the target
assert (df.casual + df.registered == df.cnt).all()

# 2. Redundant temperature columns
print("corr(temp, atemp):", round(df[["temp", "atemp"]].corr().iloc[0, 1], 4))

# 3. Rare weather category
print(df.weathersit.value_counts().sort_index())

# 4. Suspicious zeros in humidity (a humidity of exactly 0 is physically implausible)
print("rows with hum == 0:", (df.hum == 0).sum(), "on days:", df.loc[df.hum == 0, "dteday"].dt.date.unique())

# 5. Missing hours (days with < 24 rows)
hrs_per_day = df.groupby("dteday").hr.count()
print("incomplete days:", (hrs_per_day < 24).sum(), "of", len(hrs_per_day))

corr(temp, atemp): 0.9877
weathersit
1    11413
2     4544
3     1419
4        3
Name: count, dtype: int64
rows with hum == 0: 22 on days: [datetime.date(2011, 3, 10)]
incomplete days: 76 of 731


In [4]:
train_df = df[df.dteday < "2012-07-01"].copy()
val_df   = df[(df.dteday >= "2012-07-01") & (df.dteday < "2012-10-01")].copy()
test_df  = df[df.dteday >= "2012-10-01"].copy()

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {len(part):6d} rows | {part.dteday.min().date()} -> {part.dteday.max().date()} "
          f"| mean cnt: {part.cnt.mean():.1f}")
assert len(train_df) + len(val_df) + len(test_df) == len(df)

train  13003 rows | 2011-01-01 -> 2012-06-30 | mean cnt: 167.7
val     2208 rows | 2012-07-01 -> 2012-09-30 | mean cnt: 288.4
test    2168 rows | 2012-10-01 -> 2012-12-31 | mean cnt: 219.2


In [5]:
# Treat hum == 0 as a sensor defect: replace with the TRAIN median of valid readings.
# (Median computed on train only, so no information from val/test leaks in.)
hum_median = float(train_df.loc[train_df.hum > 0, "hum"].median())

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    n_bad = int((part.hum == 0).sum())
    part.loc[part.hum == 0, "hum"] = hum_median
    print(f"{name:5s}: replaced {n_bad} rows with hum = {hum_median:.2f}")

assert (train_df.hum > 0).all() and (val_df.hum > 0).all() and (test_df.hum > 0).all()

train: replaced 22 rows with hum = 0.62
val  : replaced 0 rows with hum = 0.62
test : replaced 0 rows with hum = 0.62


In [6]:
def build_features(d: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=d.index)

    # Weather: merge rare category 4 into 3; category 1 is the baseline
    w = d.weathersit.clip(upper=3)
    for k in (2, 3):
        out[f"weather_{k}"] = (w == k).astype(float)

    # Month one-hot (January is the baseline)
    for m in range(2, 13):
        out[f"month_{m}"] = (d.mnth == m).astype(float)

    # Hour-of-day profile, learned separately for working and non-working days
    # (hour 0 on a non-working day is the baseline)
    out["workingday"] = d.workingday.astype(float)
    for h in range(1, 24):
        out[f"hr_{h}_work"] = ((d.hr == h) & (d.workingday == 1)).astype(float)
        out[f"hr_{h}_off"]  = ((d.hr == h) & (d.workingday == 0)).astype(float)

    # Weather / calendar variables used as-is
    out["temp"] = d.temp
    out["hum"] = d.hum
    out["windspeed"] = d.windspeed
    out["holiday"] = d.holiday.astype(float)
    out["yr"] = d.yr.astype(float)
    return out

X_train_raw = build_features(train_df)
X_val_raw   = build_features(val_df)
X_test_raw  = build_features(test_df)

y_train = train_df.cnt.to_numpy(dtype=np.float32)
y_val   = val_df.cnt.to_numpy(dtype=np.float32)
y_test  = test_df.cnt.to_numpy(dtype=np.float32)

print(X_train_raw.shape, X_val_raw.shape, X_test_raw.shape)

(13003, 65) (2208, 65) (2168, 65)


In [7]:
#scaling on train
CONT_COLS = ["temp", "hum", "windspeed"]
scaler = StandardScaler().fit(X_train_raw[CONT_COLS])

def finalize(X_raw: pd.DataFrame) -> pd.DataFrame:
    X = X_raw.copy()
    X[CONT_COLS] = scaler.transform(X_raw[CONT_COLS])
    return X.astype(np.float32)

X_train, X_val, X_test = finalize(X_train_raw), finalize(X_val_raw), finalize(X_test_raw)
FEATURE_NAMES = list(X_train.columns)

# Sanity checks
assert not X_train.isna().any().any() and not X_test.isna().any().any()
assert list(X_val.columns) == list(X_test.columns) == FEATURE_NAMES
print("n features:", len(FEATURE_NAMES))
print(X_train[CONT_COLS].describe().loc[["mean", "std"]].round(2))   # should be ~0 and ~1

n features: 65
      temp  hum  windspeed
mean   0.0 -0.0        0.0
std    1.0  1.0        1.0


In [8]:
def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))

def mae(y, p):
    return float(np.mean(np.abs(y - p)))

def bias(y, p):
    """Mean signed error (pred - actual). Negative = under-predicting, the costly direction."""
    return float(np.mean(p - y))

def clip0(p):
    """Rentals can't be negative, so floor predictions at 0."""
    return np.clip(p, 0, None)

baseline_value = float(y_train.mean())
baseline_results = {}
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    p = np.full_like(y, baseline_value)
    baseline_results[name] = {"rmse": rmse(y, p), "mae": mae(y, p), "bias": bias(y, p)}
    print(f"baseline {name:5s} RMSE {baseline_results[name]['rmse']:.1f} | "
          f"MAE {baseline_results[name]['mae']:.1f} | bias {baseline_results[name]['bias']:.1f}")

baseline train RMSE 160.8 | MAE 126.1 | bias 0.0
baseline val   RMSE 260.7 | MAE 197.8 | bias -120.6
baseline test  RMSE 208.1 | MAE 156.6 | bias -51.5


In [9]:
joblib.dump(scaler, ARTIFACTS / "scaler.joblib")

with open(ARTIFACTS / "feature_names.json", "w") as f:
    json.dump(FEATURE_NAMES, f)

with open(ARTIFACTS / "baseline.json", "w") as f:
    json.dump({"baseline_value": baseline_value, "baseline_results": baseline_results}, f, indent=2)

# Humidity fill value (train median of valid readings), so the app can apply the same fix
with open(ARTIFACTS / "imputation.json", "w") as f:
    json.dump({"hum_median": hum_median}, f)

# Processed splits (the app uses these for plots and distributions)
np.savez(
    ARTIFACTS / "splits.npz",
    X_train=X_train.to_numpy(), y_train=y_train,
    X_val=X_val.to_numpy(),     y_val=y_val,
    X_test=X_test.to_numpy(),   y_test=y_test,
)

# All rows (train/val/test) from the ORIGINAL data, for dashboard distributions.
# Note: hum here still contains the 22 raw zeros from 2011-03-10 (before imputation).
df[["dteday", "hr", "workingday", "temp", "hum", "windspeed", "cnt"]].to_csv(
    ARTIFACTS / "raw_view.csv", index=False)

# Dates/hours of the test rows, aligned with the test predictions
test_df[["dteday", "hr"]].to_csv(ARTIFACTS / "test_index.csv", index=False)

print("saved:", sorted(p.name for p in ARTIFACTS.iterdir()))

saved: ['baseline.json', 'feature_names.json', 'hyperparams.json', 'imputation.json', 'loss_curves.npz', 'manual_torch.pt', 'module_torch.pt', 'predictions.npz', 'raw_view.csv', 'results.csv', 'scaler.joblib', 'sklearn_ridge.joblib', 'splits.npz', 'test_index.csv']


scikit learn linear model

In [10]:
from sklearn.linear_model import Ridge

alphas = [0.01, 0.1, 1, 10, 100, 1000]
alpha_val_rmse = {}
for a in alphas:
    m = Ridge(alpha=a).fit(X_train, y_train)
    alpha_val_rmse[a] = rmse(y_val, m.predict(X_val))
    print(f"alpha={a:<8} val RMSE = {alpha_val_rmse[a]:.2f}")

best_alpha = min(alpha_val_rmse, key=alpha_val_rmse.get)
print("best alpha:", best_alpha)

alpha=0.01     val RMSE = 114.02
alpha=0.1      val RMSE = 114.07
alpha=1        val RMSE = 114.60
alpha=10       val RMSE = 119.01
alpha=100      val RMSE = 140.37
alpha=1000     val RMSE = 188.10
best alpha: 0.01


In [11]:
sk_model = Ridge(alpha=best_alpha).fit(X_train, y_train)

raw_sk_test = sk_model.predict(X_test)     # keep the unclipped version just to report how many
print(f"negative test predictions before clipping: {(raw_sk_test < 0).sum()} "
      f"({(raw_sk_test < 0).mean():.2%})")

pred_sk = {
    "train": clip0(sk_model.predict(X_train)),
    "val":   clip0(sk_model.predict(X_val)),
    "test":  clip0(raw_sk_test),
}

negative test predictions before clipping: 10 (0.46%)


manual PyTorch model

In [12]:
import torch
import torch.nn as nn

torch.manual_seed(SEED)

y_mean, y_std = float(y_train.mean()), float(y_train.std())
to_t = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32)

Xt = {"train": to_t(X_train.to_numpy()), "val": to_t(X_val.to_numpy()), "test": to_t(X_test.to_numpy())}
yt_train = to_t((y_train - y_mean) / y_std).unsqueeze(1)

N_FEATURES = Xt["train"].shape[1]
LAM = best_alpha / len(y_train)      # same penalty as sklearn's alpha, per-sample form
EPOCHS = 30000
LR_GRID = [0.1, 0.3, 0.5, 0.7]

def to_bikes(pred_scaled):
    """Convert a (n,1) tensor of standardized predictions back to bikes/hour."""
    return pred_scaled.detach().squeeze(1).numpy() * y_std + y_mean

In [13]:
def train_manual(lr, epochs=EPOCHS):
    w = torch.zeros(N_FEATURES, 1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    losses = []
    for _ in range(epochs):
        pred = Xt["train"] @ w + b
        mse = ((pred - yt_train) ** 2).mean()
        loss = mse + LAM * (w ** 2).sum()        # bias is not penalized
        loss.backward()                          # autograd computes the gradients
        with torch.no_grad():                    # manual gradient-descent step
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
        losses.append(mse.item())
    return w.detach(), b.detach(), losses

manual_val = {}
for lr in LR_GRID:
    w, b, _ = train_manual(lr)
    r = rmse(y_val, to_bikes(Xt["val"] @ w + b))
    manual_val[lr] = r if np.isfinite(r) else float("inf")
    print(f"manual  lr={lr:<6} val RMSE = {manual_val[lr]:.2f}")

best_lr_manual = min(manual_val, key=manual_val.get)
w_man, b_man, losses_man = train_manual(best_lr_manual)
pred_man = {k: clip0(to_bikes(Xt[k] @ w_man + b_man)) for k in Xt}
print("best manual lr:", best_lr_manual)

manual  lr=0.1    val RMSE = 114.04
manual  lr=0.3    val RMSE = 114.02
manual  lr=0.5    val RMSE = 114.02
manual  lr=0.7    val RMSE = inf
best manual lr: 0.5


Standard PyTorch workflow model

In [14]:
class LinReg(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)
        nn.init.zeros_(self.linear.weight)       # same init as the manual version
        nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        return self.linear(x)

def train_module(lr, epochs=EPOCHS):
    model = LinReg(N_FEATURES)
    # SGD's weight_decay adds wd*w to the gradient, i.e. the gradient of (wd/2)*||w||^2.
    # So wd = 2*LAM reproduces LAM*||w||^2. No decay on the bias.
    opt = torch.optim.SGD(
        [{"params": model.linear.weight, "weight_decay": 2 * LAM},
         {"params": model.linear.bias, "weight_decay": 0.0}],
        lr=lr,
    )
    loss_fn = nn.MSELoss()
    losses = []
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(Xt["train"]), yt_train)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return model, losses

module_val = {}
for lr in LR_GRID:
    model, _ = train_module(lr)
    with torch.no_grad():
        r = rmse(y_val, to_bikes(model(Xt["val"])))
    module_val[lr] = r if np.isfinite(r) else float("inf")
    print(f"module  lr={lr:<6} val RMSE = {module_val[lr]:.2f}")

best_lr_module = min(module_val, key=module_val.get)
nn_model, losses_nn = train_module(best_lr_module)
nn_model.eval()
with torch.no_grad():
    pred_nn = {k: clip0(to_bikes(nn_model(Xt[k]))) for k in Xt}
print("best module lr:", best_lr_module)

module  lr=0.1    val RMSE = 114.04
module  lr=0.3    val RMSE = 114.02
module  lr=0.5    val RMSE = 114.02
module  lr=0.7    val RMSE = inf
best module lr: 0.5


comparison

In [15]:
rows = []
def add(name, preds):
    for split, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
        rows.append({"model": name, "split": split,
                     "RMSE": rmse(y, preds[split]), "MAE": mae(y, preds[split]), "bias": bias(y, preds[split])})

add("baseline (train mean)", {s: np.full_like(y, baseline_value) for s, y in
                              [("train", y_train), ("val", y_val), ("test", y_test)]})
add("scikit-learn Ridge", pred_sk)
add("manual PyTorch", pred_man)
add("nn.Module + optim", pred_nn)

results = pd.DataFrame(rows)
print(results.pivot(index="model", columns="split", values="RMSE").round(2)[["train", "val", "test"]])
print()
print(results.pivot(index="model", columns="split", values="bias").round(2)[["train", "val", "test"]])

# Do the three agree? Compare in original units.
coef_man = w_man.squeeze(1).numpy() * y_std
coef_nn = nn_model.linear.weight.detach().squeeze(0).numpy() * y_std
print("\nmax |coef diff| manual vs sklearn:", np.abs(coef_man - sk_model.coef_).max())
print("max |coef diff| module vs sklearn:", np.abs(coef_nn - sk_model.coef_).max())
print("max |coef diff| manual vs module :", np.abs(coef_man - coef_nn).max())
for split in ["val", "test"]:
    print(f"max |pred diff| {split} (manual vs sklearn): {np.abs(pred_man[split] - pred_sk[split]).max():.3f}")
    print(f"max |pred diff| {split} (module vs sklearn): {np.abs(pred_nn[split] - pred_sk[split]).max():.3f}")
print("final train MSE (scaled): manual", losses_man[-1], "| module", losses_nn[-1])

split                   train     val    test
model                                        
baseline (train mean)  160.78  260.65  208.06
manual PyTorch          65.09  114.02  101.67
nn.Module + optim       65.09  114.02  101.67
scikit-learn Ridge      65.09  114.02  101.67

split                  train     val   test
model                                      
baseline (train mean)   0.00 -120.64 -51.48
manual PyTorch          5.02  -16.90   6.32
nn.Module + optim       5.02  -16.90   6.32
scikit-learn Ridge      5.02  -16.90   6.32

max |coef diff| manual vs sklearn: 0.0148124695
max |coef diff| module vs sklearn: 0.014568329
max |coef diff| manual vs module : 0.0002593994
max |pred diff| val (manual vs sklearn): 0.011
max |pred diff| val (module vs sklearn): 0.011
max |pred diff| test (manual vs sklearn): 0.011
max |pred diff| test (module vs sklearn): 0.011
final train MSE (scaled): manual 0.18222744762897491 | module 0.18222744762897491


In [16]:
# Evaluate sklearn's solution under the SAME objective the PyTorch models minimize
w_sk_s = to_t(sk_model.coef_ / y_std).unsqueeze(1)
b_sk_s = to_t([(sk_model.intercept_ - y_mean) / y_std])
with torch.no_grad():
    mse_sk = ((Xt["train"] @ w_sk_s + b_sk_s - yt_train) ** 2).mean().item()
print(f"train MSE (scaled): sklearn {mse_sk:.5f} | manual {losses_man[-1]:.5f}")

# How ill-conditioned is the problem?
eig = np.linalg.eigvalsh(X_train.to_numpy().T @ X_train.to_numpy() / len(X_train))
print("largest / smallest eigenvalue:", eig.max(), "/", eig.min())

train MSE (scaled): sklearn 0.18223 | manual 0.18223
largest / smallest eigenvalue: 1.3242816 / 0.0011238396


In [17]:
# --- Trained models ---
joblib.dump(sk_model, ARTIFACTS / "sklearn_ridge.joblib")

torch.save({"w": w_man, "b": b_man, "y_mean": y_mean, "y_std": y_std},
           ARTIFACTS / "manual_torch.pt")

torch.save({"state_dict": nn_model.state_dict(), "n_features": N_FEATURES,
            "y_mean": y_mean, "y_std": y_std},
           ARTIFACTS / "module_torch.pt")

# --- Predictions (already clipped at 0), loss curves, results table ---
np.savez(ARTIFACTS / "predictions.npz",
         **{f"{m}_{s}": p[s] for m, p in [("sk", pred_sk), ("manual", pred_man), ("module", pred_nn)]
            for s in ["train", "val", "test"]})

np.savez(ARTIFACTS / "loss_curves.npz",
         manual=np.array(losses_man), module=np.array(losses_nn))

results.to_csv(ARTIFACTS / "results.csv", index=False)

# --- Hyperparameters and settings the dashboard needs ---
with open(ARTIFACTS / "hyperparams.json", "w") as f:
    json.dump({"best_alpha": best_alpha, "best_lr_manual": best_lr_manual,
               "best_lr_module": best_lr_module, "epochs": EPOCHS,
               "y_mean": y_mean, "y_std": y_std,
               "clip_predictions_at_zero": True}, f, indent=2)

print("saved:", sorted(p.name for p in ARTIFACTS.iterdir()))

saved: ['baseline.json', 'feature_names.json', 'hyperparams.json', 'imputation.json', 'loss_curves.npz', 'manual_torch.pt', 'module_torch.pt', 'predictions.npz', 'raw_view.csv', 'results.csv', 'scaler.joblib', 'sklearn_ridge.joblib', 'splits.npz', 'test_index.csv']


In [18]:
test_eval = test_df[["dteday", "hr", "workingday", "cnt"]].copy()
test_eval["pred"] = pred_sk["test"]
test_eval["err"] = test_eval.pred - test_eval.cnt

by_hr = test_eval.groupby(["workingday", "hr"]).agg(
    mean_cnt=("cnt", "mean"),
    rmse=("err", lambda e: np.sqrt((e ** 2).mean())),
    bias=("err", "mean"),
).round(1)
print(by_hr.loc[1])   # working days
print(by_hr.loc[0])   # non-working days

print("share of predictions below zero (before clipping):", (raw_sk_test < 0).mean().round(4))

    mean_cnt   rmse   bias
hr                        
0       41.9   64.9   48.6
1       17.9   66.8   56.9
2        9.0   70.6   61.2
3        5.1   65.4   55.9
4        7.4   65.2   56.7
5       34.1   58.3   52.5
6      123.7   38.9   27.9
7      353.7  114.4  -40.5
8      614.3  225.9 -132.8
9      305.0   71.9  -29.5
10     166.9   33.2   12.2
11     198.0   33.8    0.2
12     263.4   61.1  -27.9
13     252.4   53.0  -17.2
14     238.1   52.2  -18.2
15     275.5   74.9  -42.1
16     382.3  107.9  -72.6
17     618.6  211.9 -103.3
18     547.0  185.7  -59.0
19     374.0  118.3  -10.9
20     261.4   75.6   14.6
21     190.8   64.4   33.9
22     143.4   54.6   39.2
23      99.4   50.9   36.7
    mean_cnt   rmse  bias
hr                       
0       97.2   58.0  41.3
1       82.4   51.4  35.5
2       54.8   59.7  53.4
3       25.0   67.3  62.4
4        9.0   65.9  60.5
5        9.5   64.1  58.0
6       21.3   64.6  56.8
7       56.9   64.5  40.4
8      135.3   99.4  13.7
9      201.1